# Academia–Practice Interaction Mapping Using NLP  
**Notebook 08:Apply Rules to Davlan/XLM-RoBERTa Entities**

**Author:** Kamila Lewandowska  
**Project Phase:** In Progress  
**Last Updated:** July 2025  

---

## Objective

Apply the rule-based classification logic developed on the manually annotated *common entities* dataset to the remaining *Davlan-only* entities. 

---

## Workflow Summary

- Load raw Davlan output (`ner_davlan_pl.csv`) and filter out already classified entities  
- Preprocess the remaining Davlan-only entities:
  - Clean and normalize organization names  
  - Remove likely academic entities using a keyword-based filter  
  - Lemmatize names for more robust matching  
- Apply rule-based classification using pre-defined keyword lists (from Notebook 05)  
- Split into rule-matched and unmatched groups for further inspection  

---

## Rule-Based Categorization

- A dictionary of lemmatized keywords was used to match organizations to one of 11 categories  

---

## Key Outcomes

- **Davlan-only non-academic entities after cleaning:** *3149*  
- **Classified by rules:** *1104*
- **Remaining unmatched (“Other / Unclear”):** *2045*  


In [28]:
import pandas as pd
import numpy as np
from ast import literal_eval
import re
import stanza
from collections import Counter

# Data preparation and preprocessing

## Create davlan_only dataset

In [3]:
# Isolate Dalan-Only Entities

# Load full Davlan output
df_davlan_all = pd.read_csv("../output/ner_davlan_pl.csv")  
df_common = pd.read_csv("../output/common_org_entities.csv")

In [4]:
df_davlan_all.head(5)

,Text,ORG_Entities_xlm,ICS_ID
0,Badania skupiające się na szczegółowej analizi...,['Komitetu Nauk Weterynaryjnych i Rozrodu Zwie...,00153fbd-82f7-48c4-b5bd-e830bc390244
1,"Birdwatching, czyli obserwacje w terenie ptakó...","['Królewskie Towarzystwo Ochrony Ptaków', 'Zak...",002768f1-8b96-4e0f-bcc8-192eb0594e60
2,Efektywny transfer wiedzy jest podstawowym czy...,[],00500483-f00c-4410-b6f7-8650a003125f
3,Ważnym obszarem działalności naukowej WSPiA je...,"['WSPiA', 'WSPiA']",006e7fef-2083-426d-9c1b-1affd27b939e
4,Znaczna część europejskiego dziedzictwa archeo...,"['Inter', 'Archaeological Heritage Office of S...",00901439-d91a-48e0-903a-26a4253c3a0c


In [5]:
# Check the type of data of the row with entities

type(df_davlan_all["ORG_Entities_xlm"].iloc[0])

str

In [7]:
# Extract unique ORG entities from df_davlan_all

# Flatten all (ICS_ID, ORG_Entity) pairs from Davlan

# List to hold flattened records

all_davlan_entities = []

# Loop through each row in davlan data
for _, row in df_davlan_all.iterrows():
    ics_id = row["ICS_ID"]
    try:
        org_list = literal_eval(row["ORG_Entities_xlm"])
        for org in org_list:
            org_cleaned = org.strip()
            if isinstance(org_cleaned, str) and org_cleaned:
                all_davlan_entities.append((ics_id, org_cleaned))
    except:
        continue

# Create dataframe
all_davlan_flat = pd.DataFrame(all_davlan_entities, columns=["ICS_ID", "ORG_Entity"])

In [8]:
all_davlan_flat.head(20)

,ICS_ID,ORG_Entity
0,00153fbd-82f7-48c4-b5bd-e830bc390244,Komitetu Nauk Weterynaryjnych i Rozrodu Zwierz...
1,00153fbd-82f7-48c4-b5bd-e830bc390244,Rady Doradczej
2,00153fbd-82f7-48c4-b5bd-e830bc390244,Advisory Board
3,00153fbd-82f7-48c4-b5bd-e830bc390244,Med
4,00153fbd-82f7-48c4-b5bd-e830bc390244,We
5,00153fbd-82f7-48c4-b5bd-e830bc390244,ry
6,00153fbd-82f7-48c4-b5bd-e830bc390244,Journal of Applied Genetics
7,00153fbd-82f7-48c4-b5bd-e830bc390244,SPRINGER
8,00153fbd-82f7-48c4-b5bd-e830bc390244,Gene
9,00153fbd-82f7-48c4-b5bd-e830bc390244,ELSEVIER


In [20]:
# Filter out entities that were already processed (the 6111 “common”)

# Set of already classified entities
common_entities_set = set(df_common['ORG_Entity'].str.strip())

# Keep only davlan-only entries
df_davlan_only = all_davlan_flat[~all_davlan_flat["ORG_Entity"].isin(common_entities_set)].copy()

# Drop rows with duplicate ICS_ID and ORG_Entity pairs
df_davlan_only_dedup = df_davlan_only.drop_duplicates(subset=['ICS_ID', 'ORG_Entity']).copy()

# Check number of unique entities and entries
print("Unique entities:", df_davlan_only_dedup["ORG_Entity"].nunique())
print("Total rows:", len(df_davlan_only_dedup))

Unique entities: 3638
Total rows: 4495


In [21]:
# Convert to DataFrame and save

df_davlan_only_dedup.to_csv("../output/davlan_only_entities.csv", index=False)

## Preprocess davlan_only: STEP 1 Basic cleaning

In [12]:
# Function for basic claening: whitespace, removinh redundant entries

def clean_entity(entity):
    if not isinstance(entity, str):       # Check if the input is a string; if not, return None
        return None
    entity = re.sub(r'\s+', ' ', entity).strip()    # Replace multiple whitespace characters with a single space and trim leading/trailing spaces
    entity = re.sub(r'^[^\w]+', '', entity)         # Remove any non-word characters from the beginning of the string
    entity = re.sub(r'[^\w.]+$', '', entity)        # Remove any non-word characters (excluding period) from the end of the string
    return entity if len(entity) >= 3 else None      # Return the cleaned entity if it has at least 3 characters; otherwise, return None

In [23]:
# Apply the clean_entity function

df_davlan_only_dedup["Cleaned_Entity"] = df_davlan_only_dedup["ORG_Entity"].apply(clean_entity)
davlan_only_cleaned = df_davlan_only_dedup.dropna(subset=["Cleaned_Entity"]).reset_index(drop=True)
print(f"After cleaning: {len(davlan_only_cleaned)} rows.")

After cleaning: 3759 rows.


## Preprocess davlan_only: STEP 2 Rule-based academic entity removal

In [24]:
# Load academic keyword list (copied from notebook 04)

academic_keywords = [
    'instytut', 'pan', 'university', 'nauk', 'uniwersytet', 'wydział', 'wydzial', 'department', 'badań', 'akadem', 
    'katedr', 'politechni', 'laboratorium', 'research', 'institut', 'fizy', 'matematy', 
    'architektur', 'pracown', 'pedagogi', 'filozof', 'medycyn', 'medyczn', 'medical', 'językoznawstwo', 
    'mickiewicz', 'biolog', 'studia', 'uczeln', 'kolegium', 'collegium', 'studium', 'colleg', 'universit', 
    'wyższ', 'journal', 'springer', 'doktor']

In [25]:
# Lowercase for rule-based match

davlan_only_cleaned["entity_lower"] = davlan_only_cleaned["Cleaned_Entity"].str.lower()

In [26]:
# Remove academic entities

davlan_only_cleaned["Is_Academic"] = davlan_only_cleaned["entity_lower"].apply(lambda x: any(kw in x for kw in academic_keywords))
davlan_only_non_academic = davlan_only_cleaned[~davlan_only_cleaned["Is_Academic"]].copy().reset_index(drop=True)
print(f"After removing academic entities: {len(davlan_only_non_academic)} rows.")

After removing academic entities: 3149 rows.


# Rule-based Classification

In [29]:
# Download and initialize Stanza for Polish

stanza.download("pl")
nlp = stanza.Pipeline(lang="pl", processors="tokenize,mwt,lemma")

2025-07-14 16:34:48 INFO: Downloaded file to C:\Users\lewandowska\stanza_resources\resources.json
2025-07-14 16:34:48 INFO: Downloading default packages for language: pl (Polish) ...
2025-07-14 16:34:49 INFO: File exists: C:\Users\lewandowska\stanza_resources\pl\default.zip
2025-07-14 16:34:51 INFO: Finished downloading models and saved to C:\Users\lewandowska\stanza_resources
2025-07-14 16:34:51 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-07-14 16:34:51 INFO: Downloaded file to C:\Users\lewandowska\stanza_resources\resources.json
2025-07-14 16:34:51 INFO: Loading these models for language: pl (Polish):
| Processor | Package      |
----------------------------
| tokenize  | pdb          |
| mwt       | pdb          |
| lemma     | pdb_nocharlm |

2025-07-14 16:34:51 INFO: Using device: cpu
2025-07-14 16:34:51 INFO: Loading: tokenize
2025-07-14 16:34:54 INFO: Loading: mwt
2025-07-14 16:34:54 INFO: Loading: lemma
2025-07-14 16:34:54 INFO: Done loading processors!


In [30]:
# Function to lemmatize entities

def lemmatize_entity(text):
    doc = nlp(text)
    lemmas = [word.lemma for sent in doc.sentences for word in sent.words]
    return " ".join(lemmas)

In [31]:
# Apply lemmatize entities function

davlan_only_non_academic["Lemma_Entity"] = davlan_only_non_academic["Cleaned_Entity"].apply(lemmatize_entity)

In [32]:
# Fix incorrect lemmatizations (from notebook 05)

lemma_fixes = {
    "do_tar": "DG",
    "sa .": "s.a.",
    "s.a .": "s.a.",
    "o.o": "o.o.",
    "z ograniczona_odpowiedzialność .": "z o.o.",
    "z ograniczona_odpowiedzialność": "z o.o.",
    "do_spraw .": "ds.",
    "te_jeta_bowy": "TV",
    "roktóry": "rp",
    "et_yg_pały": "ETW",
    "to_sysapipiporoby": "TVP",
    "lt_yp_dwownokinoć": "Ltd",
    "fektometr_pici": "FM",
    "Bspódniczoab": "BSP"
}

# Apply the correction

def apply_fixes(text):
    for wrong, correct in lemma_fixes.items():
        text = text.replace(wrong, correct)
    return text

davlan_only_non_academic["Lemma_Cleaned"] = davlan_only_non_academic["Lemma_Entity"].apply(apply_fixes)

In [50]:
# Ruled based keyword dictionary (from notebook 05 - enhanced manually and stemmed after reviewing unmached entities)

lemmatized_keywords_by_category_clean = {
       "Government / Public Administration": [
        "urząd", "ministerstw", "minister", "gmin", "powiat", "rada", "sejm", "senat",
        "wojewódz", "rp", "komisj", "samorząd", "narodow", "krajow", "izb",
        "państw", "miast", "regional", "marszałkow", "urzęd", "narod", "kraj", "ministr",
        "sąd", "inspektorat", "parlament", "rzeczpospolit", "agencj", "stołeczn", "ambasad",
        "rzecznik", "senack", "inspek", "konsul", "turyst", "delegatur",
        "punkt", "fundusz", "skarb", "rząd", "dyrekcj", "archiw", "sztab", "ris", "krrit"
    ],
     "NGO / Association / Foundation": [
        "fundacj", "stowarzyszen", "towarzystw", "zrzeszen", "federacj", "koalicj", "association", "związk", "obywatel",
         "wspólot", "pomoc", "nno"
    ],
   
    "Media / Publishing": [
        "radi", "tv", "gazet", "media", "wydawnictw", "czasopism", "pras", "tvp", "fm", "YouTube"
    ],
    "Cultural Institution / Arts": [
        "muze", "teatr", "galer", "festiwal", "filharmoni", "dom", "kultur", "sztuk", "artystycz",
        "twórcz", "książ", "muzycz", "museum", "koncert", "zamek", "królewsk", "pałac", "klub", "filmow",
        "heritage"
    ],
    "Health / Hospitals / Medical": [
        "zdrow", "klinik", "szpital", "lekarz", "medycz", "przychodni", "sanatori", "rehabilitacj",
        "hospicj", "chorob", "epidemi", "health", "uzdrowisk", "chory", "onkolog"
    ],
    "Religious Organization": [
        "kościół", "parafi", "diecezj", "episkopat", "zakon", "misja", "cerkiew", "salwatorian", "duchow", "biblijn", "kuria",
        "metropolit"
    ],
    "Military / Defense / Security": [
        "wojsk", "żandarmeri", "bezpiecz", "policj", "straż", "obron", "militar", "komendant", "lotnicz", "zbrojny", "bsp"
    ],
    "International Organization / EU": [
        "europejsk", "unia", "ue", "nato", "unesco", "oecd", "who", "międzynarod", "international", "european", "DG", "union", "dyrektoriat", "nations",
        "światow", "unijn", "eu-xfel", "europe", "eurostat"
    ],
    "Company / Business": [
       "s.a.", "s.a", "z o.o.", "hold", "firm", "grupa", "przedsiębiorstw", "msp", "csr",
        "technolog", "logistyk", "consulting", "solutions", "commerc", "industr", "group", "spółk", "Ltd", "stoczni", "kopaln", "huta"
    ],
       
    "Education (non-university)": [
        "liceum", "licea", "technikum", "szkoł", "podstawow", "przedszkol", "edukacj", "bibliotek", "szkolen", "nauczyciel", "podyplomow",
        "oświat", "kurator", "kuratori", "szkol"
    ],
    "Other / Unclear": []  # fallback category
}


In [51]:
# Define a function to match a lemmatized entity against category keyword lists

def match_entity_to_category(entity, keyword_dict):
    """
    Matches a lemmatized entity string to one of the predefined categories
    based on the presence of any lemmatized keywords.

    Parameters:
        entity (str): The lemmatized name of an organization.
        keyword_dict (dict): Dictionary mapping category names to keyword lists.

    Returns:
        str: The matched category name, or "Other / Unclear" if no match found.
    """
    entity_lower = entity.lower()
    for category, keywords in keyword_dict.items():
        if any(keyword in entity_lower for keyword in keywords):
            return category
    return "Other / Unclear"

In [52]:
# Apply the match_entity_to_category function

davlan_only_non_academic["Matched_Category"] = davlan_only_non_academic["Lemma_Cleaned"].apply(lambda x: match_entity_to_category(x, lemmatized_keywords_by_category_clean))

In [53]:
# Split: rule-matched and unmatched

davlan_non_academic_rule_matched = davlan_only_non_academic[davlan_only_non_academic["Matched_Category"] != "Other / Unclear"].copy()
davlan_non_academic_unmatched = davlan_only_non_academic[davlan_only_non_academic["Matched_Category"] == "Other / Unclear"].copy()

print(f"Rule-matched: {len(davlan_non_academic_rule_matched)} rows.")
print(f"Unmatched {len(davlan_non_academic_unmatched)} rows.")

Rule-matched: 1104 rows.
Unmatched 2045 rows.


In [54]:
print(davlan_non_academic_rule_matched["ORG_Entity"].nunique())
print(davlan_non_academic_unmatched["ORG_Entity"].nunique())


965
1839


In [55]:
# Inspect the data: check frequencies of categories

cat_freq = Counter(davlan_non_academic_rule_matched["Matched_Category"])
cat_freq

Counter({'Government / Public Administration': 433,
         'Company / Business': 133,
         'Cultural Institution / Arts': 133,
         'NGO / Association / Foundation': 91,
         'International Organization / EU': 89,
         'Education (non-university)': 72,
         'Media / Publishing': 59,
         'Health / Hospitals / Medical': 53,
         'Military / Defense / Security': 25,
         'Religious Organization': 16})

In [56]:
# Display entity names and lemmas for the most frequent unmatched entries 

unmatched_with_originals = (
    davlan_non_academic_unmatched
    .groupby(["Lemma_Cleaned", "ORG_Entity"])
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

unmatched_with_originals.head(50)

,Lemma_Cleaned,ORG_Entity,count
214,YouTube,YouTube,23
1663,web of science,Web of Science,10
1243,polityka,Polityka,9
1744,zespo,Zespo,7
738,grup,Grup,6
1443,spo,spo,6
146,Polska,Polski,5
1375,science,Science,5
744,główny,Główne,5
851,jedno,Jedno,5


In [59]:
# Export files

# Export full matched entities
davlan_non_academic_rule_matched.to_csv("../output/davlan_non_academic_rule_matched.csv", index=False)

# Export full unmatched entities
davlan_non_academic_unmatched.to_csv("../output/davlan_non_academic_unmatched.csv", index=False)

# Create annotation file with duplicates (one row per occurrence)
davlan_for_annotation = (
    davlan_non_academic_unmatched[["ICS_ID", "ORG_Entity"]]
    .copy()
)
davlan_for_annotation["Matched_Category"] = ""

# Export to CSV
davlan_for_annotation.to_csv("../output/davlan_non_academic_for_annotation.csv", index=False)
